In [13]:
from pathlib import Path

current_dir = Path.cwd()
# .parent moves up one level from 'notebooks' to 'CLV_Project'
project_root = current_dir.parent

print(f"Project Root: {project_root}")

Project Root: c:\Users\sneha\OneDrive\Documents\CLV_Project


In [37]:
import pandas as pd
import numpy as np

# # load data
transactions = pd.read_csv(f"{project_root}/Data/Transaction.csv" , encoding="latin1")
customers = pd.read_csv(f"{project_root}/Data/Customer.csv")

transactions.head()
# customers.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [40]:
transactions.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [41]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17049 entries, 0 to 17048
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order_ID                  17049 non-null  object 
 1   Customer_ID               17049 non-null  object 
 2   Date                      17049 non-null  object 
 3   Age                       17049 non-null  int64  
 4   Gender                    17049 non-null  object 
 5   City                      17049 non-null  object 
 6   Product_Category          17049 non-null  object 
 7   Unit_Price                17049 non-null  float64
 8   Quantity                  17049 non-null  int64  
 9   Discount_Amount           17049 non-null  float64
 10  Total_Amount              17049 non-null  float64
 11  Payment_Method            17049 non-null  object 
 12  Device_Type               17049 non-null  object 
 13  Session_Duration_Minutes  17049 non-null  int64  
 14  Pages_

In [45]:
transactions["InvoiceDate"] = pd.to_datetime(
    transactions["InvoiceDate"],
    dayfirst=True
)

In [46]:
print("Missing CustomerIDs:", transactions["CustomerID"].isna().sum())

# remove rows without customer
transactions = transactions.dropna(subset=["CustomerID"])

Missing CustomerIDs: 135080


In [47]:
transactions = transactions[transactions["Quantity"] > 0]
transactions = transactions[transactions["UnitPrice"] > 0]

In [48]:
transactions["order_value"] = (
    transactions["Quantity"] * transactions["UnitPrice"]
)

In [49]:
transactions = transactions.rename(columns={
    "InvoiceNo": "order_id",
    "StockCode": "product_id",
    "Description": "product",
    "Quantity": "quantity",
    "InvoiceDate": "order_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country"
})

In [50]:
print("Unique customers:", transactions["customer_id"].nunique())
print("Unique orders:", transactions["order_id"].nunique())
print("Total revenue:", transactions["order_value"].sum())
print("Avg order value:", transactions["order_value"].mean())

print(
    "Date range:",
    transactions["order_date"].min(),
    "to",
    transactions["order_date"].max()
)

Unique customers: 4338
Unique orders: 18532
Total revenue: 8911407.904
Avg order value: 22.396999889415003
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [51]:
orders_per_customer = (
    transactions.groupby("customer_id")["order_id"]
    .nunique()
)

orders_per_customer.describe()

count    4338.000000
mean        4.272015
std         7.697998
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: order_id, dtype: float64

In [52]:
revenue_per_customer = (
    transactions.groupby("customer_id")["order_value"]
    .sum()
)

revenue_per_customer.describe()

count      4338.000000
mean       2054.266460
std        8989.230441
min           3.750000
25%         307.415000
50%         674.485000
75%        1661.740000
max      280206.020000
Name: order_value, dtype: float64

In [53]:
output_path = f"{project_root}/Data/transactions_cleaned.csv"

transactions.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: c:\Users\sneha\OneDrive\Documents\CLV_Project/Data/transactions_cleaned.csv
